<a href="https://colab.research.google.com/github/alangu-hep/data-driven-habitability/blob/main/habitability_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
Let's start by importing external libraries
'''

import xml.etree.ElementTree as ET, urllib.request, gzip, io # OEC Access

import numpy as np # Number Operations
import matplotlib.pyplot as plt # Plots & Graphs
from astropy.constants import G, sigma_sb, M_earth, R_earth, GM_earth, M_jup, L_sun, au # Astropy gives us the solar system's constants

In [ ]:
'''
Project Variables
'''

oec_url = "https://github.com/OpenExoplanetCatalogue/oec_gzip/raw/master/systems.xml.gz" # If we want to use a different data format, we can change the URL

# Names in line with the tag names in the OEC
system_properties = ['distance']
planetary_properties = ['discoverymethod', 'mass', 'radius', 'semimajoraxis', 'temperature', 'period']
star_properties = ['temperature']

In [ ]:
oec = ET.parse(gzip.GzipFile(fileobj=io.BytesIO(urllib.request.urlopen(oec_url).read())))

In [ ]:
'''
Equations for ESI 1
'''

def flux(distance, radius, temperature, luminosity = None):
  '''
  Distance is to be processed first (either semimajor axis or an approximation)
  Radius & Temperature are stellar
  Luminosity is to be calculated unless specified otherwise
  '''
  if luminosity is not None:
    return luminosity/(4*np.pi*distance**2) # This ends the function early, so we don't need an else statement--this is called an early return
  return (radius**2 * sigma_sb.value * temperature**4)/(distance**2) # The ** notation means exponent

def semimajoraxis(period, m_star, m_planet):
  return np.cbrt((period**2 * G.value * (m_star+m_planet))/(4*np.pi**2))

def esi_one(stellar_flux, solar_flux, radius, earth_radius):
  esi = 1 - np.sqrt(0.5 * (((stellar_flux-solar_flux)/(stellar_flux+solar_flux))**2 + ((radius-earth_radius)/(radius+earth_radius))**2))
  return esi

'''
Equations for ESI 2
'''

def density(mass, radius):
  volume = 4/3 * np.pi * radius**3
  return mass/volume

def escape_velocity(mass, radius, gm = None):
  if gm is not None:
    return np.sqrt((2*gm)/radius)
  return np.sqrt((2*G.value*mass)/radius)

def esi_two(properties, values):
  '''
  Takes properties as dictionaries with the format:
  {'property': wt}
  and values as dictionaries with the format:
  {'property': {'value': float, 'earth': float}}
  '''

  if not isinstance(properties, dict):
    raise TypeError("Properties must be a dictionary")
    return

  if not isinstance(values, dict):
    raise TypeError("Values must be a dictionary")
    return

  indexes = []

  for property, weight in properties.items():
    print(f'Creating index for {property}')
    val = values[property]['value']
    earth_val = values[property]['earth']
    index = (1-abs((val - earth_val)/(val + earth_val)))**weight
    indexes.append(index)

  return np.prod(indexes)